# 单种子 curriculum 训练(proactive RR / D63)

**目标**:先在**一个种子、一个 config** 上,用 **curriculum 由易到难**把完成率磨到 ~100%(像 RR:先能不丢车,再谈优化),然后再谈矩阵/消融。

**为什么 curriculum**(D63):数据诊断出完成率低的真因是**入区放行饿死**——车堆在入区口进不来(96–98% 没完成的车从没进区,区内淤死仅 2%,死锁/碰撞 0),而选项都在、信号员用同样选项能到 1.0 → 这是**可学的放行时机/顺序**问题。繁忙日一上来就堵死、学不到东西,所以**先喂清闲日学会放行,再逐步升到高峰**。

**规则**:按每天车流量分 4 档累加(≤185→≤286→≤300→全部 = 9→33→48→64 天),当前档 **train 完成率≥0.9 或到 cap** 就升档,**最后跑满高峰**(守 D27/D28/D43);**环境/奖励/网络一律不动,只改喂哪些天**。

**功能**:逐 iter 实时进度(含档位)、每评估周期 **train + val 完成率都打印/记录**、`--run-tag`+`--resume` 断点续训(12h 被杀重跑本格即续)。


## 设置(服务器前置)


In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project

PROJ = '/rds/homes/h/hxn886/ondemand/final_project'
PY   = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX = (f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
          f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK. shell 用  !{PREFIX} <script> <args>')


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '(本流程不需要 GPU,CPU 即可)')


## 前置:full-panel BC 权重(热启动源)


In [ ]:
from pathlib import Path
p = Path('artifacts/dataset/bc_fullpanel_gnn.pt')
print('bc_fullpanel_gnn.pt', 'OK' if p.exists() else 'MISSING —— 先在 bc_fullpanel_server.ipynb 训出', p)


## 0) Smoke(先跑通:小 iters、快升档,验证 curriculum + 进度 + 续训)
**在服务器上跑**(要 torch)。先看:热启动体检动作构成正常?每档能升?train/val 都打印?


In [ ]:
!{PREFIX} scripts/13_train_ppo.py --encoder gnn \
    --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 0.5 \
    --train-eval-days 3 --iters 10 --steps-per-iter 512 --eval-every 2 --critic-warmup-iters 2 \
    --early-stop-patience 0 --run-tag smoke_cur_gnn


## 1) 正式:单种子 curriculum 训练(重跑本格 = 断点续训)
`--iters` 可调;`early-stop` 只在最终档生效。12h 被杀后**重跑本格**即从断点续(连档位恢复)。


In [ ]:
!{PREFIX} scripts/13_train_ppo.py --encoder gnn \
    --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999  --promote-at 0.9 \
    --train-eval-days 6 --iters 80 --steps-per-iter 2048 --eval-every 5 \
    --early-stop-patience 6 --run-tag cur_gnn_schedule_s0 --resume


## 2) 结果:train / val 完成率随 iter(两套都记录)
跑完读 `<tag>.json`,跑一半读 `<tag>_last.pt`。


In [ ]:
import json, torch
from pathlib import Path
tag='cur_gnn_schedule_s0'; rep=Path('artifacts/reports')
evals=None; best=None
if (rep/f'{tag}.json').exists():
    d=json.loads((rep/f'{tag}.json').read_text()); evals=d['evals']; best=d['best']
elif (rep/f'{tag}_last.pt').exists():
    st=torch.load(rep/f'{tag}_last.pt', map_location='cpu', weights_only=False); evals=st['evals']; best=st['best']
if not evals:
    print('还没有评估结果(run 还没到第一个 eval 周期)')
else:
    print(f"{'iter':>4} {'tier':>4} {'train完成':>9} {'val完成':>8} {'val_reward':>11}  WAIT/set/cxl")
    for e in evals:
        tc=e.get('train_completion'); tcs=(f'{tc:.3f}' if tc is not None else '  -  ')
        print(f"{e['iter']:>4} {e.get('tier',0)+1:>4} {tcs:>9} {e['eval_completion_rate']:>8.3f} "
              f"{e['eval_episode_reward']:>11.0f}  {e['eval_frac_wait']:.0%}/{e['eval_frac_set']:.0%}/{e['eval_frac_cancel']:.0%}")
    print(f"\n最佳(val): it{best['iter']} 完成率 {best['completion']:.3f} reward {best['reward']:.1f}")
    try:
        import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
        its=[e['iter'] for e in evals]
        plt.figure(figsize=(7,3))
        plt.plot(its,[e['eval_completion_rate'] for e in evals],'-o',label='val')
        plt.plot(its,[e.get('train_completion') for e in evals],'-s',label='train(cur tier)')
        plt.axhline(1.0,ls='--',c='g',lw=0.8,label='signaller=1.0'); plt.ylim(0,1.05)
        plt.xlabel('iter'); plt.ylabel('completion'); plt.legend(); plt.title(tag); plt.tight_layout()
        plt.savefig(rep/f'{tag}_curve.png',dpi=110); plt.show(); print('saved', rep/f'{tag}_curve.png')
    except Exception as ex: print('plot skipped:', ex)


## 说明
- **进度**:训练格逐 iter 打印(档位/完成率/EV/选路KL/早停);每 5 轮打印 train+val 完成率 + 是否升档。
- **续训**:中断/12h 被杀 → 重跑训练格(已完成的不重训,从 `<tag>_last.pt` 续,档位一起恢复)。
- **产物**:`artifacts/reports/cur_gnn_schedule_s0.json`(曲线/最佳)+ `_best.pt`(最佳权重)+ `_last.pt`(续训全状态)。
- **test 不碰**(D43):训练只用 train/val;要 test 的 headline,单独加 `--final-test`(结束后 best 在 test 评一次)。
